# Data Description

- Author: Codex
- Date first created: 20261013
- Background: 這是原始假訂單資料清理後的版本。原本有 122 列，刪除 2 列重複紀錄、3 筆件數缺值和 2 筆負單價後，剩下 115 筆。另新增訂單金額，供後續分析使用。
- Unit of observation: 每列是一筆訂單，`order_id` 不重複。
- Sample period: 2026-09-01 至 2026-09-30。
- Known issues: 資料為人工生成，只用來練習整理與分析流程，不能用來描述真實消費行為。
- Definition for each variable:
    - `order_id`：訂單編號。
    - `order_date`：訂單日期，格式為 YYYY-MM-DD。
    - `category`：商品類別，已統一為 A、B、C，沒有前後空白。
    - `quantity`：購買件數，保留的資料皆為 1 至 5 件的整數。
    - `unit_price_twd`：每件商品的單價，單位為新臺幣元，負值已排除。
    - `order_amount_twd`：訂單金額，等於購買件數乘以單價，單位為新臺幣元。
- Data path: [data/processed/demo_orders_clean.csv](demo_orders_clean.csv)
- Source data: [原始 CSV](../raw/demo_orders.csv)；[原始資料說明](../raw/demo_orders_documentation.ipynb)
- Processing code: [data/code/demo_orders_processing.ipynb](../code/demo_orders_processing.ipynb)
- Role: `analysis`
- Verification status: 已確認 115 筆訂單皆無缺值、編號不重複，日期、類別、件數與單價符合設定，訂單金額計算正確。
- SHA-256: `566d03f1701b16128fd119cc2776d775787816fb13f3d7e6d799988f6ce43104`
- Downstream analysis input: `data/processed/demo_orders_clean.csv`；[分析 notebook](../../finding/demo_orders_analysis.ipynb)

要重建資料，先在專案根目錄執行 `python data/code/demo_generate_orders.py`，再執行上方的處理 notebook。

In [1]:
from pathlib import Path
import hashlib

import pandas as pd
from IPython.display import display

# 從 notebook 所在位置往上找專案，從根目錄開啟也能執行。
folder = Path.cwd().resolve()
for project_root in [folder, *folder.parents]:
    if (project_root / "AGENTS.md").is_file() and (project_root / "templates").is_dir():
        break
else:
    raise FileNotFoundError("找不到專案根目錄，請從專案內開啟 notebook。")

## Change for Your Own Data

下方讀取已儲存的整理後 CSV，並比對檔案指紋。若還需要調整清理方式，回到處理 notebook 修改，再重新產生資料。

In [2]:
data_path = (project_root / "data/processed/demo_orders_clean.csv").resolve()
expected_hash = "566d03f1701b16128fd119cc2776d775787816fb13f3d7e6d799988f6ce43104"

if data_path.parent != project_root / "data/processed":
    raise ValueError("資料路徑超出預期目錄。")
if not data_path.is_file():
    raise FileNotFoundError(f"找不到資料：{data_path.name}，請先產生資料。")

# 確認讀到的是這份文件記錄的資料版本。
file_hash = hashlib.sha256(data_path.read_bytes()).hexdigest()
if file_hash != expected_hash:
    raise ValueError("資料版本已改變，請先核對來源與說明文件。")

## Summary

整理後共有 115 筆、6 個欄位，比原始資料多了訂單金額。以下確認存出的資料與處理結果一致。

In [3]:
df = pd.read_csv(data_path)
expected_columns = [
    "order_id", "order_date", "category", "quantity", "unit_price_twd",
    "order_amount_twd",
]
assert list(df.columns) == expected_columns, "欄位與處理結果不一致。"
assert len(df) == 115, "整理後資料應有 115 筆。"
print(f"資料共有 {len(df)} 筆、{len(df.columns)} 欄。")

資料共有 115 筆、6 欄。


### Sample

以下是前 10 筆整理後的訂單。例如第一筆買了 1 件、單價 80 元，訂單金額為 80 元；第二筆買了 3 件、單價 97 元，金額為 291 元。

In [4]:
df.head(10)

,order_id,order_date,category,quantity,unit_price_twd,order_amount_twd
0,D0001,2026-09-01,A,1,80,80
1,D0002,2026-09-02,B,3,97,291
2,D0003,2026-09-03,C,5,114,570
3,D0004,2026-09-04,A,2,131,262
4,D0005,2026-09-05,B,4,148,592
5,D0006,2026-09-06,C,1,165,165
6,D0007,2026-09-07,A,3,182,546
7,D0008,2026-09-08,B,5,199,995
8,D0010,2026-09-10,A,4,233,932
9,D0011,2026-09-11,B,1,250,250


### Summary Stats

115 筆訂單的平均購買件數約為 3.01 件，平均金額約為 539.75 元。訂單金額從 80 元到 1,420 元不等，所有欄位都沒有缺值。

In [5]:
print(f"整理後資料摘要（N={len(df)}）")
display(df.describe(include="all"))

column_info = pd.DataFrame({
    "型態": df.dtypes.astype(str),
    "缺值數": df.isna().sum(),
})
print("欄位型態與缺值數")
display(column_info)

整理後資料摘要（N=115）


,order_id,order_date,category,quantity,unit_price_twd,order_amount_twd
count,115,115,115,115.000000,115.000000,115.000000
unique,115,30,3,NaN,NaN,NaN
top,D0001,2026-09-01,A,NaN,NaN,NaN
freq,1,4,39,NaN,NaN,NaN
mean,NaN,NaN,NaN,3.008696,179.191304,539.747826
std,NaN,NaN,NaN,1.435732,64.829181,337.452553
min,NaN,NaN,NaN,1.000000,80.000000,80.000000
25%,NaN,NaN,NaN,2.000000,122.500000,262.000000
50%,NaN,NaN,NaN,3.000000,182.000000,485.000000
75%,NaN,NaN,NaN,4.000000,233.000000,750.000000


欄位型態與缺值數


,型態,缺值數
order_id,object,0
order_date,object,0
category,object,0
quantity,int64,0
unit_price_twd,int64,0
order_amount_twd,int64,0


### Assertions

最後用程式檢查筆數、欄位與訂單編號，並逐筆確認「訂單金額＝件數 × 單價」。若任何一項不符合預期，程式會在這裡停止。

In [6]:
assert df["order_id"].is_unique, "訂單編號有重複。"
assert not df.isna().any().any(), "整理後資料仍有缺值。"
assert set(df["category"]) == {"A", "B", "C"}
assert df["quantity"].between(1, 5).all()
assert (df["quantity"] % 1 == 0).all()
assert df["unit_price_twd"].between(80, 300).all()

expected_amount = df["quantity"] * df["unit_price_twd"]
assert (df["order_amount_twd"] == expected_amount).all(), "訂單金額計算有誤。"

dates = pd.to_datetime(df["order_date"], format="%Y-%m-%d")
assert dates.between("2026-09-01", "2026-09-30").all()
assert hashlib.sha256(data_path.read_bytes()).hexdigest() == expected_hash
print("筆數、欄位、訂單編號、缺值、範圍與金額公式皆通過檢查。")

筆數、欄位、訂單編號、缺值、範圍與金額公式皆通過檢查。
